In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F
import time
from pyspark.sql.functions import col, when, to_date, year, month, dayofmonth, trim, lower, round as spark_round,avg,count
import matplotlib.pyplot as plt
import numpy as np

In [19]:
import time
import os
import shutil
from pyspark.sql import SparkSession

# ============================================
# SETUP
# ============================================
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("CSV vs Parquet") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

# Verify configuration
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Master URL: {spark.conf.get('spark.master')}")


Driver Memory: 8g
Shuffle Partitions: 8
Master URL: local[4]


In [20]:
df = spark.read.parquet("data/clean/")
df.cache()

DataFrame[Flow Duration: int, Total Fwd Packets: int, Total Backward Packets: int, Total Length of Fwd Packets: int, Total Length of Bwd Packets: int, Fwd Packet Length Max: int, Fwd Packet Length Min: int, Fwd Packet Length Mean: double, Fwd Packet Length Std: double, Bwd Packet Length Max: int, Bwd Packet Length Min: int, Bwd Packet Length Mean: double, Bwd Packet Length Std: double, Flow Bytes/s: double, Flow Packets/s: double, Flow IAT Mean: double, Flow IAT Std: double, Flow IAT Max: int, Flow IAT Min: int, Fwd IAT Total: int, Fwd IAT Mean: double, Fwd IAT Std: double, Fwd IAT Max: int, Fwd IAT Min: int, Bwd IAT Total: int, Bwd IAT Mean: double, Bwd IAT Std: double, Bwd IAT Max: int, Bwd IAT Min: int, Fwd Header Length34: bigint, Bwd Header Length: int, Fwd Packets/s: double, Bwd Packets/s: double, Min Packet Length: int, Max Packet Length: int, Packet Length Mean: double, Packet Length Std: double, Packet Length Variance: double, FIN Flag Count: int, SYN Flag Count: int, RST Flag

In [21]:
string_cols = [c for c, t in df.dtypes if t == "string"]
print(string_cols)

['Label']


In [22]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="Label",
    outputCol="label_index",
    handleInvalid="keep"
)

df1 = indexer.fit(df).transform(df)

df1.select("Label", "label_index").show(10)

+----------+-----------+
|     Label|label_index|
+----------+-----------+
|     Recon|        3.0|
|  DoS/DDoS|        5.0|
|  DoS/DDoS|        5.0|
|  DoS/DDoS|        5.0|
|  DoS/DDoS|        5.0|
|    Normal|        4.0|
|BruteForce|        0.0|
|    Normal|        4.0|
| WebAttack|        1.0|
|    Normal|        4.0|
+----------+-----------+
only showing top 10 rows



In [23]:
# --- VectorAssembler ---
from pyspark.ml.feature import VectorAssembler
# Exclude the target label and any string columns from features
exclude_cols = set(string_cols + ["label_index"])
feature_cols = [c for c in df1.columns if c not in exclude_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="keep"   # skips rows with null/NaN in any feature col
)
df2 = assembler.transform(df1)


In [7]:
df2.select("Label", "label_index", "features_raw").show(10, truncate=True)

+----------+-----------+--------------------+
|     Label|label_index|        features_raw|
+----------+-----------+--------------------+
|     Recon|        3.0|(58,[0,1,2,3,4,5,...|
|  DoS/DDoS|        5.0|[8.5666313E7,5.0,...|
|  DoS/DDoS|        5.0|(58,[0,1,3,5,6,7,...|
|  DoS/DDoS|        5.0|[1.01312683E8,9.0...|
|  DoS/DDoS|        5.0|(58,[0,1,2,3,5,6,...|
|    Normal|        4.0|[155443.0,14.0,11...|
|BruteForce|        0.0|[9048607.0,9.0,15...|
|    Normal|        4.0|[175.0,2.0,2.0,84...|
| WebAttack|        1.0|(58,[0,1,2,14,15,...|
|    Normal|        4.0|[59535.0,3.0,4.0,...|
+----------+-----------+--------------------+
only showing top 10 rows



In [24]:
# --- StandardScaler ---
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,   # zero-centre (dense output)
    withStd=True
)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)
# Show scaled features alongside label
df3.select("Label", "label_index", "features").show(10, truncate=True)

+----------+-----------+--------------------+
|     Label|label_index|            features|
+----------+-----------+--------------------+
|     Recon|        3.0|[-0.4972899797216...|
|  DoS/DDoS|        5.0|[2.06637466670124...|
|  DoS/DDoS|        5.0|[-0.3612588173649...|
|  DoS/DDoS|        5.0|[2.53461077016710...|
|  DoS/DDoS|        5.0|[-0.4968528180669...|
|    Normal|        4.0|[-0.4926391219778...|
|BruteForce|        0.0|[-0.2265006848296...|
|    Normal|        4.0|[-0.4972857002777...|
| WebAttack|        1.0|[-0.3319497738909...|
|    Normal|        4.0|[-0.4955092821627...|
+----------+-----------+--------------------+
only showing top 10 rows



In [25]:
from pyspark.ml.stat import Summarizer
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Compute std for each numeric column individually
from pyspark.sql import functions as F

# Filter out columns with zero std or any nulls/NaN
good_cols = []
for c in feature_cols:
    stats = df1.select(
        F.stddev(F.col(c)).alias("std"),
        F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1)).alias("null_count")
    ).first()
    if stats["std"] is not None and stats["std"] > 0 and stats["null_count"] == 0:
        good_cols.append(c)

print(f"Kept {len(good_cols)} / {len(feature_cols)} columns after variance/null check")

Kept 56 / 58 columns after variance/null check


In [26]:
assembler = VectorAssembler(
    inputCols=good_cols,
    outputCol="features_raw",
    handleInvalid="skip"    # drop any remaining bad rows
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

df2 = assembler.transform(df1)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)

# Update feature_cols to reflect what's actually in the vector
feature_cols = good_cols

In [27]:
from pyspark.ml.classification import RandomForestClassifier
import pandas as pd

rf_selector = RandomForestClassifier(
    featuresCol="features",
    labelCol="label_index",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf_selector.fit(df3)

# Extract importances
importance_df = pd.DataFrame({
    "feature": good_cols,   # use good_cols, not feature_cols
    "importance": rf_model.featureImportances.toArray()
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(importance_df.head(20))
print(f"\nTotal features: {len(good_cols)}")

                        feature  importance
0           Average Packet Size    0.070850
1            Packet Length Mean    0.052658
2        Bwd Packet Length Mean    0.048527
3          Avg Bwd Segment Size    0.048040
4             Bwd Header Length    0.046464
5   Total Length of Fwd Packets    0.045932
6   Total Length of Bwd Packets    0.038094
7         Bwd Packet Length Max    0.036619
8         Fwd Packet Length Max    0.032090
9        Total Backward Packets    0.031841
10         min_seg_size_forward    0.030182
11            Max Packet Length    0.028965
12                Flow IAT Mean    0.025562
13         Avg Fwd Segment Size    0.022738
14        Bwd Packet Length Std    0.022155
15            Total Fwd Packets    0.020920
16        Bwd Packet Length Min    0.020851
17                Bwd Packets/s    0.020733
18                  Fwd IAT Std    0.020683
19          Fwd Header Length55    0.020514

Total features: 56


In [28]:
# Cumulative 95% importance
importance_df["cumulative"] = importance_df["importance"].cumsum()
selected_features = importance_df[importance_df["cumulative"] <= 0.95]["feature"].tolist()

print(f"Selected {len(selected_features)} / {len(good_cols)} features")
print(importance_df[["feature", "importance", "cumulative"]])

Selected 40 / 56 features
                        feature    importance  cumulative
0           Average Packet Size  7.084967e-02    0.070850
1            Packet Length Mean  5.265771e-02    0.123507
2        Bwd Packet Length Mean  4.852708e-02    0.172034
3          Avg Bwd Segment Size  4.804033e-02    0.220075
4             Bwd Header Length  4.646362e-02    0.266538
5   Total Length of Fwd Packets  4.593203e-02    0.312470
6   Total Length of Bwd Packets  3.809438e-02    0.350565
7         Bwd Packet Length Max  3.661876e-02    0.387184
8         Fwd Packet Length Max  3.209041e-02    0.419274
9        Total Backward Packets  3.184110e-02    0.451115
10         min_seg_size_forward  3.018182e-02    0.481297
11            Max Packet Length  2.896486e-02    0.510262
12                Flow IAT Mean  2.556225e-02    0.535824
13         Avg Fwd Segment Size  2.273795e-02    0.558562
14        Bwd Packet Length Std  2.215460e-02    0.580717
15            Total Fwd Packets  2.092000e-02 

In [29]:
assembler_sel = VectorAssembler(
    inputCols=selected_features,
    outputCol="features_raw_sel",
    handleInvalid="skip"
)

scaler_sel = StandardScaler(
    inputCol="features_raw_sel",
    outputCol="features_selected",
    withMean=True,
    withStd=True
)

df2_sel = assembler_sel.transform(df1)
df_sel = scaler_sel.fit(df2_sel).transform(df2_sel)

print(f"Final: {df_sel.count()} rows x {len(selected_features)} features")

Final: 203666 rows x 40 features


In [33]:
df_sel.limit(5).toPandas().head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,label_index,features_raw_sel,features_selected
0,1092,9,6,3150,3152,1575,0,350.000000,694.509719,1576,...,0,0,0.0,0.0,0,0,Normal,0.0,"[813.8429005, 704.585067, 525.3333333, 525.333...","[0.4967251336276392, 0.5669242091779436, 0.292..."
1,579225,132,150,160,320799,160,0,1.212121,13.926212,4344,...,0,0,0.0,0.0,0,0,Normal,0.0,"[831.8348267, 1227.920672, 2138.66, 2138.66, 4...","[0.5171512065357831, 1.3589756764652234, 2.843..."
2,24368732,11,6,686,560,168,0,62.363636,80.651439,172,...,0,0,0.0,0.0,0,0,Normal,0.0,"[77.75517132, 78.24287643, 93.33333333, 93.333...","[-0.3389486822765582, -0.38102438504181635, -0..."
3,171,1,1,6,6,6,6,6.000000,0.000000,6,...,0,0,0.0,0.0,0,0,Normal,0.0,"(0.0, 0.0, 6.0, 6.0, 6.0, 9.0, 0.0, 1.0, 6.0, ...","[-0.4272234328903938, -0.4994424412357167, -0...."
4,2941634,47,46,2664,6954,456,0,56.680851,104.767466,976,...,0,0,0.0,0.0,0,0,Normal,0.0,"[307.3801341, 231.2838508, 151.173913, 151.173...","[-0.07825751563126083, -0.14940183319034522, -..."


In [30]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[
    indexer,          # StringIndexer
    assembler_sel,    # VectorAssembler (40 cols)
    scaler_sel        # StandardScaler
])

pipeline_model = pipeline.fit(df)
final_df = pipeline_model.transform(df).select(
    "features_selected", "label_index", "Label"
)

In [31]:
final_df.printSchema()
final_df.show(5, truncate=True)
print("Total rows:", final_df.count())
print("Feature vector size:", len(final_df.head(1)[0]["features_selected"]))

root
 |-- features_selected: vector (nullable = true)
 |-- label_index: double (nullable = false)
 |-- Label: string (nullable = true)

+--------------------+-----------+--------+
|   features_selected|label_index|   Label|
+--------------------+-----------+--------+
|[-0.4697783331091...|        3.0|   Recon|
|[1.88423493256545...|        5.0|DoS/DDoS|
|[-0.4638358559993...|        5.0|DoS/DDoS|
|[1.46197241335230...|        5.0|DoS/DDoS|
|[-0.4702537312779...|        5.0|DoS/DDoS|
+--------------------+-----------+--------+
only showing top 5 rows

Total rows: 203666
Feature vector size: 40


In [32]:
final_df.write.mode("overwrite").parquet("data/features/")

In [33]:
spark.stop()